In [1]:
import cv2
import numpy as np

In [5]:
window_size = 400
images = {
    "Image_1": cv2.imread("fisheye_15/fisheye_229_fov.jpg"),
    "Image_ref": cv2.imread("ohlf.png")
}
ref_name = "Image_ref"
input_images = [k for k in images if k != ref_name]
saved_pairs = {name: {"image": [], "ref": []} for name in input_images}
current_round = {name: None for name in images}
zoom = {name: 1.0 for name in images}

def clamp(v, a, b):
    return max(a, min(v, b))

def get_view(name, img):
    h, w = img.shape[:2]
    cx = cv2.getTrackbarPos("X", name)
    cy = cv2.getTrackbarPos("Y", name)
    zw, zh = int(w / zoom[name]), int(h / zoom[name])
    cx = clamp(cx, zw // 2, w - zw // 2)
    cy = clamp(cy, zh // 2, h - zh // 2)
    ox = cx - zw // 2
    oy = cy - zh // 2
    return ox, oy, zw, zh

# MOUSE CALLBACK
def mouse_cb(event, x, y, flags, param):
    name, img = param
    ox, oy, zw, zh = get_view(name, img)
    u = int(ox + x / window_size * zw)
    v = int(oy + y / window_size * zh)
    if event == cv2.EVENT_LBUTTONDOWN:
        current_round[name] = (u, v)
    if event == cv2.EVENT_RBUTTONDOWN:
        current_round[name] = None
    if event == cv2.EVENT_MOUSEWHEEL:
        zoom[name] = clamp(zoom[name] * (1.2 if flags > 0 else 1 / 1.2), 1.0, 10.0)

# WINDOWS
for name, img in images.items():
    h, w = img.shape[:2]
    cv2.namedWindow(name)
    cv2.resizeWindow(name, window_size, window_size)
    cv2.createTrackbar("X", name, w // 2, w, lambda x: None)
    cv2.createTrackbar("Y", name, h // 2, h, lambda x: None)
    cv2.setMouseCallback(name, mouse_cb, (name, img))

while True:
    for name, img in images.items():
        ox, oy, zw, zh = get_view(name, img)
        crop = img[oy:oy + zh, ox:ox + zw]
        disp = cv2.resize(crop, (window_size, window_size))
        if name == ref_name:
            for img_name in saved_pairs:
                for u, v in saved_pairs[img_name]["ref"]:
                    if ox <= u <= ox + zw and oy <= v <= oy + zh:
                        dx = int((u - ox) / zw * window_size)
                        dy = int((v - oy) / zh * window_size)
                        cv2.circle(disp, (dx, dy), 4, (0, 255, 0), -1)

        if name in saved_pairs:
            for u, v in saved_pairs[name]["image"]:
                if ox <= u <= ox + zw and oy <= v <= oy + zh:
                    dx = int((u - ox) / zw * window_size)
                    dy = int((v - oy) / zh * window_size)
                    cv2.circle(disp, (dx, dy), 4, (0, 255, 0), -1)

        if current_round[name] is not None:
            u, v = current_round[name]
            dx = int((u - ox) / zw * window_size)
            dy = int((v - oy) / zh * window_size)
            cv2.circle(disp, (dx, dy), 6, (0, 0, 255), -1)

        cv2.imshow(name, disp)

    key = cv2.waitKey(20) & 0xFF

    if key == ord("s"):
        ref_pt = current_round[ref_name]
        if ref_pt is not None:
            for img_name in input_images:
                if current_round[img_name] is not None:
                    saved_pairs[img_name]["image"].append(current_round[img_name])
                    saved_pairs[img_name]["ref"].append(ref_pt)

        for k in current_round:
            current_round[k] = None

    if key == ord("q"):
        for name, data in saved_pairs.items():
            src = np.array(data["image"], np.float32)
            dst = np.array(data["ref"], np.float32)
            print(name, data)
            if len(src) >= 4:
                h, mask = cv2.findHomography(src, dst, cv2.RANSAC)
                print(f"Homography for {name}:\n{h}\n")
            else:
                print(f"Not enough points for {name}\n")
        break

    if key == 27:
        break

cv2.destroyAllWindows()


Image_1 {'image': [(1697, 1515), (1643, 1487), (1796, 1130), (1548, 995), (1345, 1096), (1312, 1143), (1251, 1209), (1234, 1233), (1153, 1375), (1100, 1550), (1075, 1898), (1011, 2029), (1235, 2173), (1384, 2200), (1445, 2075)], 'ref': [(1302, 578), (1305, 590), (1211, 593), (1210, 685), (1257, 720), (1268, 723), (1287, 731), (1295, 730), (1327, 730), (1364, 724), (1423, 687), (1449, 687), (1448, 619), (1445, 573), (1421, 576)]}
Homography for Image_1:
[[-2.43229184e-01  2.93317061e-01  1.19626407e+03]
 [-3.24129531e-01 -6.24119645e-02  1.18698319e+03]
 [-1.10512320e-04  8.79198669e-05  1.00000000e+00]]



In [4]:
h1 = np.load("Homography for 229_camera_ 1.npy")
h2 = np.load("Homography for 229_camera_.npy")
print(h1)
print(h2)

[[-3.80508158e-01  2.30400538e-01  1.20383711e+03]
 [-3.44393073e-01 -5.12837286e-02  1.06783855e+03]
 [-2.39590097e-04  6.73004069e-05  1.00000000e+00]]
[[-3.80508158e-01  2.30400538e-01  1.20383711e+03]
 [-3.44393073e-01 -5.12837286e-02  1.06783855e+03]
 [-2.39590097e-04  6.73004069e-05  1.00000000e+00]]


In [6]:
print(h)

[[-2.43229184e-01  2.93317061e-01  1.19626407e+03]
 [-3.24129531e-01 -6.24119645e-02  1.18698319e+03]
 [-1.10512320e-04  8.79198669e-05  1.00000000e+00]]


In [8]:
np.save('cam_229_homo.npy',h)

In [9]:
h3 = np.load('cam_229_homo.npy')

In [10]:
print(h3)

[[-2.43229184e-01  2.93317061e-01  1.19626407e+03]
 [-3.24129531e-01 -6.24119645e-02  1.18698319e+03]
 [-1.10512320e-04  8.79198669e-05  1.00000000e+00]]
